# Feature Engineering — First 24-Hour Features

This notebook builds a model-ready feature DataFrame for the ICU mortality prediction task. For each of the 1,424 patients in the analytic cohort, we extract features from their first 24 hours of ICU stay.

## Feature categories

1. **Demographics** — age, gender, BMI (already in cohort table)
2. **Vital signs** — heart rate, blood pressure, respiratory rate, SpO2, temperature (aggregated min/max/mean over first 24 hours)
3. **Laboratory values** — creatinine, glucose, WBC, lactate, sodium, potassium, hemoglobin, bilirubin (aggregated over first 24 hours)
4. **Apache components** — Glasgow Coma Scale, admission diagnosis category

## Output

A DataFrame `features` with one row per patient, ~50 feature columns, and a binary target `in_hospital_mortality`. Saved to `data/features.parquet` (gitignored — do not commit).

## Setup: imports and database connection

In [26]:
import sys
sys.path.append("/Users/azadeh_sereshki/Documents/Projects/eicu-mortality-prediction/src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from db import get_engine
from sqlalchemy import text

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

engine = get_engine()
print("Connected. Ready to build features.")

Connected. Ready to build features.


## Pull demographics from the database

In [2]:
# Pull cohort + hospital characteristics into one query
# The cohort view already has age_numeric and all patient-level columns
# We join with the hospital table to get bed category, teaching status, region

demographics_query = """
    SELECT 
        c.patientunitstayid,
        c.patienthealthsystemstayid,
        c.age_numeric AS age,
        c.gender,
        c.admissionheight,
        c.admissionweight,
        c.unittype,
        c.hospitalid,
        h.numbedscategory,
        h.teachingstatus,
        h.region,
        c.hospitaldischargestatus
    FROM cohort c
    LEFT JOIN hospital h ON c.hospitalid = h.hospitalid
"""

demo = pd.read_sql(demographics_query, engine)
print(f"Shape: {demo.shape}")
print(f"Expected: (1424, 12)")
demo.head()

Shape: (1424, 12)
Expected: (1424, 12)


,patientunitstayid,patienthealthsystemstayid,age,gender,admissionheight,admissionweight,unittype,hospitalid,numbedscategory,teachingstatus,region,hospitaldischargestatus
0,147784,134042,60,Female,154.9,95.6,Med-Surg ICU,67,NaN,False,Midwest,Alive
1,151179,136669,59,Female,149.9,NaN,Med-Surg ICU,66,100 - 249,False,Midwest,Expired
2,151867,137216,44,Male,172.7,NaN,SICU,68,<100,False,Midwest,Alive
3,151900,137239,66,Female,165.1,86.8,MICU,73,>= 500,True,Midwest,Alive
4,152954,138053,41,Female,170.2,81.0,Med-Surg ICU,71,100 - 249,False,Midwest,Alive


## Compute BMI and the target variable

In [3]:
# Compute BMI from height (cm) and weight (kg)
# BMI = weight / (height/100)^2

demo["bmi"] = demo["admissionweight"] / ((demo["admissionheight"] / 100) ** 2)

# Create binary mortality target
# 1 = died in hospital, 0 = discharged alive
demo["in_hospital_mortality"] = (demo["hospitaldischargestatus"] == "Expired").astype(int)

# Quick validation
print("BMI summary statistics:")
print(demo["bmi"].describe())
print(f"\nMortality rate: {demo['in_hospital_mortality'].mean() * 100:.2f}%")
print(f"Deaths: {demo['in_hospital_mortality'].sum()} of {len(demo)}")

BMI summary statistics:
count    1356.000000
mean       30.221011
std        26.848874
min         2.291667
25%        23.369203
50%        27.526569
75%        32.793041
max       860.969388
Name: bmi, dtype: float64

Mortality rate: 8.29%
Deaths: 118 of 1424


## Check data quality on BMI

In [4]:
# Check for implausible BMI values
print("BMI value distribution:")
print(f"  BMI < 12 (implausibly low):  {(demo['bmi'] < 12).sum()} patients")
print(f"  BMI 12-60 (plausible):       {((demo['bmi'] >= 12) & (demo['bmi'] <= 60)).sum()} patients")
print(f"  BMI > 60 (implausibly high): {(demo['bmi'] > 60).sum()} patients")
print(f"  BMI missing:                 {demo['bmi'].isna().sum()} patients")

# Show extreme values for sanity-checking
print("\nMost extreme BMI values (likely data entry errors):")
print(demo.nlargest(5, "bmi")[["patientunitstayid", "admissionheight", "admissionweight", "bmi"]])
print()
print(demo.nsmallest(5, "bmi")[["patientunitstayid", "admissionheight", "admissionweight", "bmi"]])

BMI value distribution:
  BMI < 12 (implausibly low):  1 patients
  BMI 12-60 (plausible):       1342 patients
  BMI > 60 (implausibly high): 13 patients
  BMI missing:                 68 patients

Most extreme BMI values (likely data entry errors):
      patientunitstayid  admissionheight  admissionweight         bmi
1293            3135484             33.6             97.2  860.969388
1006            2684922             58.8            110.1  318.443704
1352            3157608             71.0            100.0  198.373339
471             1143992             67.9             90.0  195.210404
300              842498            167.6            515.0  183.340833

      patientunitstayid  admissionheight  admissionweight        bmi
226              482789            600.0             82.5   2.291667
372             1054428            167.0             35.0  12.549751
184              393867            187.9             44.7  12.660591
1218            3012414            170.2             

## Clean implausible values and finalize demographics

In [6]:
# Clean implausible BMI values
# Clinical plausibility range: 12 <= BMI <= 60
# Values outside this range are almost certainly data entry errors
demo["bmi_missing"] = demo["bmi"].isna() | (demo["bmi"] < 12) | (demo["bmi"] > 60)
demo.loc[(demo["bmi"] < 12) | (demo["bmi"] > 60), "bmi"] = np.nan

# Clean implausible height (should be 100-220cm for adults)
# Keep valid heights, null the rest
demo.loc[(demo["admissionheight"] < 100) | (demo["admissionheight"] > 220), "admissionheight"] = np.nan

# Clean implausible weight (should be 30-300kg for adults)
demo.loc[(demo["admissionweight"] < 30) | (demo["admissionweight"] > 300), "admissionweight"] = np.nan

# Encode gender as binary (1 = Male, 0 = Female, NaN for anything else)
demo["gender_male"] = (demo["gender"] == "Male").astype(int)

# Encode teaching status as 0/1 (keep NaN for missing)
demo["teaching_hospital"] = demo["teachingstatus"].astype(float)  # True→1.0, False→0.0, NaN→NaN

# Confirm the cleanup
print(f"BMI after cleanup:")
print(demo["bmi"].describe())
print(f"\nMissing BMI: {demo['bmi'].isna().sum()} patients (was {68}, now includes the 14 implausible cases)")
print(f"\nFeature preview:")

display_cols = ["patientunitstayid", "age", "gender_male", "bmi", "bmi_missing", "unittype", "numbedscategory", "teaching_hospital", "in_hospital_mortality"]
demo[display_cols].head()

BMI after cleanup:
count    1342.000000
mean       28.685017
std         7.713483
min        12.549751
25%        23.266711
50%        27.460454
75%        32.564220
max        58.906298
Name: bmi, dtype: float64

Missing BMI: 82 patients (was 68, now includes the 14 implausible cases)

Feature preview:


,patientunitstayid,age,gender_male,bmi,bmi_missing,unittype,numbedscategory,teaching_hospital,in_hospital_mortality
0,147784,60,0,39.843278,False,Med-Surg ICU,NaN,0.0,0
1,151179,59,0,NaN,True,Med-Surg ICU,100 - 249,0.0,1
2,151867,44,1,NaN,True,SICU,<100,0.0,0
3,151900,66,0,31.843851,False,MICU,>= 500,1.0,0
4,152954,41,0,27.961850,False,Med-Surg ICU,100 - 249,0.0,0


## Build the final demographics DataFrame

In [8]:
# Create the features DataFrame
# This is the base that we'll join vitals, labs, and Apache data onto

features = demo[[
    "patientunitstayid",        # primary key — used to join in more features later
    "age",
    "gender_male",
    "admissionheight",
    "admissionweight",
    "bmi",
    "bmi_missing",
    "unittype",
    "numbedscategory",
    "teaching_hospital",
    "region",
    "in_hospital_mortality",    # the target
]].copy()

print(f"Features DataFrame shape: {features.shape}")
print(f"\nColumn list:")

for col in features.columns:
    n_missing = features[col].isna().sum()
    dtype = features[col].dtype
    print(f"  {col:30s} {str(dtype):15s} missing: {n_missing}")

Features DataFrame shape: (1424, 12)

Column list:
  patientunitstayid              int64           missing: 0
  age                            int64           missing: 0
  gender_male                    int64           missing: 0
  admissionheight                float64         missing: 35
  admissionweight                float64         missing: 55
  bmi                            float64         missing: 82
  bmi_missing                    bool            missing: 0
  unittype                       str             missing: 0
  numbedscategory                str             missing: 211
  teaching_hospital              float64         missing: 0
  region                         str             missing: 127
  in_hospital_mortality          int64           missing: 0


## Check the raw teachingstatus values I pulled from the database

In [9]:
print("Raw teachingstatus value counts (from the demographics query):")
print(demo["teachingstatus"].value_counts(dropna=False))
print(f"\nNulls in teachingstatus: {demo['teachingstatus'].isna().sum()}")

# Check if teaching_hospital has the expected missing values
print(f"\nNulls in teaching_hospital (after astype(float)): {demo['teaching_hospital'].isna().sum()}")

# And check the same for region
print(f"\nRegion value counts:")
print(demo["region"].value_counts(dropna=False))

Raw teachingstatus value counts (from the demographics query):
teachingstatus
False    1277
True      147
Name: count, dtype: int64

Nulls in teachingstatus: 0

Nulls in teaching_hospital (after astype(float)): 0

Region value counts:
region
Midwest      465
South        454
West         284
NaN          127
Northeast     94
Name: count, dtype: int64


In [12]:
import os
os.makedirs("../data", exist_ok=True)

features.to_parquet("../data/features_demographics.parquet", index=False)
print(f"Saved {len(features)} rows × {len(features.columns)} columns to data/features_demographics.parquet")

test_reload = pd.read_parquet("../data/features_demographics.parquet")
print(f"Reload verified: {test_reload.shape}")

Saved 1424 rows × 12 columns to data/features_demographics.parquet
Reload verified: (1424, 12)


## Query heart rate for cohort patients in first 24 hours

In [13]:
# Pull heart rate measurements for cohort patients during first 24 hours
# vitalperiodic has ~1.6M rows total; this query filters to just the relevant subset
# observationoffset = minutes from ICU admission (0 to 1440 = first 24hr)

hr_query = """
    SELECT 
        vp.patientunitstayid,
        vp.observationoffset,
        vp.heartrate
    FROM vitalperiodic vp
    INNER JOIN cohort c ON vp.patientunitstayid = c.patientunitstayid
    WHERE 
        vp.observationoffset >= 0
        AND vp.observationoffset <= 1440
        AND vp.heartrate IS NOT NULL
        AND vp.heartrate > 0          -- physiologically impossible values
"""

print("Querying heart rate data... this may take 10-30 seconds due to the large table size")

hr_raw = pd.read_sql(hr_query, engine)

print(f"\nRetrieved {len(hr_raw):,} heart rate measurements for {hr_raw['patientunitstayid'].nunique()} patients")
print(f"Expected cohort size: 1,424 patients")
print(f"\nPer-patient measurement counts:")
print(hr_raw.groupby("patientunitstayid").size().describe())

Querying heart rate data... this may take 10-30 seconds due to the large table size

Retrieved 375,569 heart rate measurements for 1370 patients
Expected cohort size: 1,424 patients

Per-patient measurement counts:
count    1370.000000
mean      274.137956
std        31.248818
min         3.000000
25%       275.000000
50%       285.000000
75%       287.000000
max       289.000000
dtype: float64


## Value distribution

In [14]:
# Examine heart rate value distribution to identify outliers
print("Heart rate value distribution:")
print(hr_raw["heartrate"].describe())

# Plausible clinical range: 20-300 bpm
# Below 20 = cardiac arrest or dead patient (artifact in data)
# Above 300 = measurement error or device malfunction
print(f"\nImplausible heart rate values:")
print(f"  < 20 bpm:  {(hr_raw['heartrate'] < 20).sum():,} readings")
print(f"  20-300:    {((hr_raw['heartrate'] >= 20) & (hr_raw['heartrate'] <= 300)).sum():,} readings")
print(f"  > 300 bpm: {(hr_raw['heartrate'] > 300).sum():,} readings")

Heart rate value distribution:
count    375569.000000
mean         85.166728
std          19.116015
min           9.000000
25%          71.000000
50%          83.000000
75%          98.000000
max         202.000000
Name: heartrate, dtype: float64

Implausible heart rate values:
  < 20 bpm:  3 readings
  20-300:    375,566 readings
  > 300 bpm: 0 readings


## Aggregate to per-patient features

In [15]:
# Filter to plausible heart rate range and aggregate to per-patient features
# Use .agg() to compute multiple statistics at once

hr_clean = hr_raw[(hr_raw["heartrate"] >= 20) & (hr_raw["heartrate"] <= 300)]

hr_features = (
    hr_clean.groupby("patientunitstayid")["heartrate"]
    .agg([
        ("hr_min", "min"),
        ("hr_max", "max"),
        ("hr_mean", "mean"),
        ("hr_count", "count"),
    ])
    .reset_index()
)

print(f"Heart rate features computed for {len(hr_features)} patients")
print(f"\nSummary of per-patient aggregates:")
print(hr_features[["hr_min", "hr_max", "hr_mean", "hr_count"]].describe())
print(f"\nFirst 5 patients:")
hr_features.head()

Task was destroyed but it is pending!
task: <Task pending name='Task-3706' coro=<_async_in_context.<locals>.run_in_context() done, defined at /opt/anaconda3/envs/eicu/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-3707' coro=<Kernel.shell_main() running at /opt/anaconda3/envs/eicu/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /opt/anaconda3/envs/eicu/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/opt/anaconda3/envs/eicu/lib/python3.11/site-packages/numpy/_core/numeric.py:1480: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  axis = tuple(normalize_axis_index(ax, ndim, argname) for ax in axis)
Task was destroyed but it is pending!
task: <Task pending name='Task-3707' coro=<Kernel.shell_main() running at /opt/anaconda3/envs/eicu/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>


Heart rate features computed for 1370 patients

Summary of per-patient aggregates:
            hr_min       hr_max      hr_mean     hr_count
count  1370.000000  1370.000000  1370.000000  1370.000000
mean     69.696350   109.678832    85.077344   274.135766
std      15.857604    22.464820    16.868615    31.250966
min      20.000000    47.000000    37.000000     3.000000
25%      59.000000    93.000000    72.850063   275.000000
50%      69.000000   107.000000    83.258935   285.000000
75%      80.000000   124.000000    97.245260   287.000000
max     124.000000   202.000000   139.937063   289.000000

First 5 patients:


,patientunitstayid,hr_min,hr_max,hr_mean,hr_count
0,147784,79,109,93.720280,286
1,151179,82,171,98.978873,284
2,151867,91,100,94.742537,268
3,151900,64,118,84.947917,288
4,152954,74,113,89.637324,284


## Join heart rate features back to main features DataFrame

In [16]:
# Join heart rate features onto the main features DataFrame
# Use LEFT JOIN so patients without HR data keep their rows (as NaN)
features = features.merge(hr_features, on="patientunitstayid", how="left")

# Add a missingness indicator — important signal for the model
features["hr_missing"] = features["hr_mean"].isna()

print(f"Features DataFrame shape after HR join: {features.shape}")
print(f"Expected: (1424, 17)  -- 12 demographics + 4 HR features + 1 missingness flag")

print(f"\nPatients missing heart rate data: {features['hr_missing'].sum()}")
print(f"(Should be ~54 patients who had no HR measurements)")

# Sanity check on mortality rate across HR coverage
print(f"\nMortality rate by HR availability:")
print(features.groupby("hr_missing")["in_hospital_mortality"].agg(["count", "mean"]))

Features DataFrame shape after HR join: (1424, 17)
Expected: (1424, 17)  -- 12 demographics + 4 HR features + 1 missingness flag

Patients missing heart rate data: 54
(Should be ~54 patients who had no HR measurements)

Mortality rate by HR availability:
            count      mean
hr_missing                 
False        1370  0.084672
True           54  0.037037


## Save and checkpoint

In [17]:
# Save the current features DataFrame including heart rate
features.to_parquet("../data/features_with_hr.parquet", index=False)
print(f"Saved features ({features.shape}) to data/features_with_hr.parquet")

# Print a quick summary for the README
print(f"\nFeature columns:")
for col in features.columns:
    n_missing = features[col].isna().sum()
    print(f"  {col:30s} missing: {n_missing}")

Saved features ((1424, 17)) to data/features_with_hr.parquet

Feature columns:
  patientunitstayid              missing: 0
  age                            missing: 0
  gender_male                    missing: 0
  admissionheight                missing: 35
  admissionweight                missing: 55
  bmi                            missing: 82
  bmi_missing                    missing: 0
  unittype                       missing: 0
  numbedscategory                missing: 211
  teaching_hospital              missing: 0
  region                         missing: 127
  in_hospital_mortality          missing: 0
  hr_min                         missing: 54
  hr_max                         missing: 54
  hr_mean                        missing: 54
  hr_count                       missing: 54
  hr_missing                     missing: 0


## Pull all five vitals in one query

In [18]:
# Pull all vital signs for cohort patients during first 24 hours
# Single query is much faster than 5 separate queries

vitals_query = """
    SELECT 
        vp.patientunitstayid,
        vp.observationoffset,
        vp.heartrate,
        vp.systemicsystolic,
        vp.systemicdiastolic,
        vp.systemicmean,
        vp.respiration,
        vp.sao2,
        vp.temperature
    FROM vitalperiodic vp
    INNER JOIN cohort c ON vp.patientunitstayid = c.patientunitstayid
    WHERE 
        vp.observationoffset >= 0
        AND vp.observationoffset <= 1440
"""

print("Querying all vital signs... this may take 30-60 seconds")

vitals_raw = pd.read_sql(vitals_query, engine)

print(f"\nRetrieved {len(vitals_raw):,} rows")
print(f"Unique patients: {vitals_raw['patientunitstayid'].nunique()}")
print(f"\nNon-null counts per vital:")

for col in ["heartrate", "systemicsystolic", "systemicdiastolic", "systemicmean", "respiration", "sao2", "temperature"]:
    n = vitals_raw[col].notna().sum()
    pct = 100 * n / len(vitals_raw)
    print(f"  {col:20s} {n:>9,} non-null ({pct:.1f}%)")

Querying all vital signs... this may take 30-60 seconds


Task was destroyed but it is pending!
task: <Task pending name='Task-3845' coro=<_async_in_context.<locals>.run_in_context() done, defined at /opt/anaconda3/envs/eicu/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-3846' coro=<Kernel.shell_main() running at /opt/anaconda3/envs/eicu/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /opt/anaconda3/envs/eicu/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/opt/anaconda3/envs/eicu/lib/python3.11/site-packages/sqlalchemy/engine/cursor.py:1197: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  rows = dbapi_cursor.fetchall()
Task was destroyed but it is pending!
task: <Task pending name='Task-3846' coro=<Kernel.shell_main() running at /opt/anaconda3/envs/eicu/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>
Task was destroyed but it is pending!
task: <Task p


Retrieved 377,644 rows
Unique patients: 1378

Non-null counts per vital:
  heartrate              375,621 non-null (99.5%)
  systemicsystolic        51,859 non-null (13.7%)
  systemicdiastolic       51,859 non-null (13.7%)
  systemicmean            52,320 non-null (13.9%)
  respiration            332,181 non-null (88.0%)
  sao2                   337,731 non-null (89.4%)
  temperature             23,632 non-null (6.3%)


## Define plausibility ranges for each vital

In [19]:
# Define clinically plausible ranges for each vital sign
# Values outside these ranges are almost always data entry errors or device artifacts

vital_ranges = {
    "heartrate":         (20, 300),    # bpm
    "systemicsystolic":  (40, 300),    # mmHg
    "systemicdiastolic": (20, 200),    # mmHg
    "systemicmean":      (30, 200),    # mmHg
    "respiration":       (4, 60),      # breaths per minute
    "sao2":              (50, 100),    # percentage
    "temperature":       (28, 43),     # Celsius
}

# Check how many readings are out of range for each
print("Implausible readings per vital:")
for col, (lo, hi) in vital_ranges.items():
    raw = vitals_raw[col].dropna()
    too_low = (raw < lo).sum()
    too_high = (raw > hi).sum()
    in_range = ((raw >= lo) & (raw <= hi)).sum()
    pct_clean = 100 * in_range / len(raw) if len(raw) > 0 else 0
    print(f"  {col:20s} too_low: {too_low:>5,}  too_high: {too_high:>5,}  in_range: {in_range:>9,} ({pct_clean:.2f}%)")

Task was destroyed but it is pending!
task: <Task pending name='Task-3901' coro=<_async_in_context.<locals>.run_in_context() done, defined at /opt/anaconda3/envs/eicu/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-3902' coro=<Kernel.shell_main() running at /opt/anaconda3/envs/eicu/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /opt/anaconda3/envs/eicu/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/opt/anaconda3/envs/eicu/lib/python3.11/tokenize.py:529: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  pseudomatch = _compile(PseudoToken).match(line, pos)
Task was destroyed but it is pending!
task: <Task pending name='Task-3902' coro=<Kernel.shell_main() running at /opt/anaconda3/envs/eicu/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>


Implausible readings per vital:
  heartrate            too_low:    55  too_high:     0  in_range:   375,566 (99.99%)
  systemicsystolic     too_low:   267  too_high:     8  in_range:    51,584 (99.47%)
  systemicdiastolic    too_low:   223  too_high:    68  in_range:    51,568 (99.44%)
  systemicmean         too_low:   150  too_high:   180  in_range:    51,990 (99.37%)
  respiration          too_low: 2,848  too_high:   333  in_range:   329,000 (99.04%)
  sao2                 too_low:   679  too_high:     0  in_range:   337,052 (99.80%)
  temperature          too_low:    50  too_high:   845  in_range:    22,737 (96.21%)


## Aggregate all vitals to per-patient features

In [20]:
# Aggregate each vital to per-patient features (min, max, mean, count)
# Filter to plausible range first, then aggregate

vital_prefixes = {
    "heartrate":         "hr",
    "systemicsystolic":  "sbp",
    "systemicdiastolic": "dbp",
    "systemicmean":      "map",
    "respiration":       "rr",
    "sao2":              "spo2",
    "temperature":       "temp",
}

# Drop the heart rate columns we already have (we'll re-merge fresh)
hr_cols = ["hr_min", "hr_max", "hr_mean", "hr_count", "hr_missing"]
features = features.drop(columns=[c for c in hr_cols if c in features.columns])

# Build aggregates for each vital and merge
for col, prefix in vital_prefixes.items():
    lo, hi = vital_ranges[col]
    
    # Filter to plausible range
    valid = vitals_raw[(vitals_raw[col] >= lo) & (vitals_raw[col] <= hi)][["patientunitstayid", col]]
    
    # Aggregate
    agg = (
        valid.groupby("patientunitstayid")[col]
        .agg([
            (f"{prefix}_min", "min"),
            (f"{prefix}_max", "max"),
            (f"{prefix}_mean", "mean"),
            (f"{prefix}_count", "count"),
        ])
        .reset_index()
    )
    
    # Merge into features
    features = features.merge(agg, on="patientunitstayid", how="left")
    features[f"{prefix}_missing"] = features[f"{prefix}_mean"].isna()
    
    n_present = (~features[f"{prefix}_missing"]).sum()
    print(f"  {prefix:5s} ({col:20s}): {n_present}/1424 patients have data ({100*n_present/1424:.1f}%)")

print(f"\nFeatures DataFrame shape: {features.shape}")
print(f"Expected: (1424, 47)  -- 12 demographics + 7 vitals × 5 cols")

  hr    (heartrate           ): 1370/1424 patients have data (96.2%)
  sbp   (systemicsystolic    ): 246/1424 patients have data (17.3%)
  dbp   (systemicdiastolic   ): 245/1424 patients have data (17.2%)
  map   (systemicmean        ): 245/1424 patients have data (17.2%)
  rr    (respiration         ): 1259/1424 patients have data (88.4%)
  spo2  (sao2                ): 1362/1424 patients have data (95.6%)
  temp  (temperature         ): 108/1424 patients have data (7.6%)

Features DataFrame shape: (1424, 47)
Expected: (1424, 47)  -- 12 demographics + 7 vitals × 5 cols


## Quick sanity check on the new features

In [21]:
# Sanity check: per-patient mean values for each vital should be in normal clinical ranges
vital_summary_cols = [f"{p}_mean" for p in vital_prefixes.values()]

print("Per-patient mean values across cohort:")
print(features[vital_summary_cols].describe().T[["mean", "std", "min", "50%", "max"]])

Per-patient mean values across cohort:
                 mean        std        min         50%         max
hr_mean     85.077344  16.868615  37.000000   83.258935  139.937063
sbp_mean   121.660562  20.013364  72.000000  119.200522  256.000000
dbp_mean    60.146337  10.845345  33.968421   59.120155   91.119048
map_mean    79.890942  11.532497  56.566434   79.000000  119.310469
rr_mean     19.758801   4.620154   6.538462   19.014778   45.770642
spo2_mean   96.689972   2.147583  77.937238   96.869425  100.000000
temp_mean   37.020419   0.917429  32.977095   37.230372   38.529329


## Save and commit

In [22]:
# Save the full vitals feature set
features.to_parquet("../data/features_with_vitals.parquet", index=False)
print(f"Saved features ({features.shape}) to data/features_with_vitals.parquet")

# Commit-ready summary
print(f"\nFinal column count: {len(features.columns)}")
print(f"Patients with all 7 vitals: {(~features[[f'{p}_missing' for p in vital_prefixes.values()]].any(axis=1)).sum()}")

Saved features ((1424, 47)) to data/features_with_vitals.parquet

Final column count: 47
Patients with all 7 vitals: 47


## Pull cuff BP and intermittent temperature from vitalaperiodic

In [23]:
# Pull intermittent (non-continuous) vitals from vitalaperiodic
# This table holds cuff BP measurements and intermittent temperature
# Most patients have these recorded every 1-4 hours

aperiodic_query = """
    SELECT 
        va.patientunitstayid,
        va.observationoffset,
        va.noninvasivesystolic,
        va.noninvasivediastolic,
        va.noninvasivemean
    FROM vitalaperiodic va
    INNER JOIN cohort c ON va.patientunitstayid = c.patientunitstayid
    WHERE 
        va.observationoffset >= 0
        AND va.observationoffset <= 1440
"""

print("Querying aperiodic vitals (cuff BP)...")
aperiodic_raw = pd.read_sql(aperiodic_query, engine)

print(f"\nRetrieved {len(aperiodic_raw):,} rows")
print(f"Unique patients with aperiodic data: {aperiodic_raw['patientunitstayid'].nunique()}")
print(f"\nNon-null counts:")

for col in ["noninvasivesystolic", "noninvasivediastolic", "noninvasivemean"]:
    n = aperiodic_raw[col].notna().sum()
    pct = 100 * n / len(aperiodic_raw)
    print(f"  {col:25s} {n:>9,} non-null ({pct:.1f}%)")

Querying aperiodic vitals (cuff BP)...

Retrieved 73,854 rows
Unique patients with aperiodic data: 1354

Non-null counts:
  noninvasivesystolic          66,267 non-null (89.7%)
  noninvasivediastolic         66,274 non-null (89.7%)
  noninvasivemean              66,689 non-null (90.3%)


## Check temperature in nursecharting

In [28]:
# nursecharting stores temperature with several possible labels
# Check what's available for cohort patients

temp_check_query = text("""
    SELECT 
        nursingchartcelltypevallabel,
        nursingchartcelltypevalname,
        COUNT(*) AS n_readings
    FROM nursecharting nc
    INNER JOIN cohort c ON nc.patientunitstayid = c.patientunitstayid
    WHERE 
        nc.nursingchartoffset >= 0
        AND nc.nursingchartoffset <= 1440
        AND (
            LOWER(nc.nursingchartcelltypevallabel) LIKE '%temp%'
            OR LOWER(nc.nursingchartcelltypevalname) LIKE '%temp%'
        )
    GROUP BY nursingchartcelltypevallabel, nursingchartcelltypevalname
    ORDER BY n_readings DESC
    LIMIT 10
""")

print("Checking temperature variables in nursecharting...")
with engine.connect() as conn:
    temp_labels = pd.read_sql(temp_check_query, conn)
print(temp_labels)

Checking temperature variables in nursecharting...
  nursingchartcelltypevallabel nursingchartcelltypevalname  n_readings
0                  Temperature             Temperature (C)       10227
1                  Temperature             Temperature (F)       10227
2                  Temperature        Temperature Location        6325


## Pull the merged BP and temperature data

In [29]:
# ===== Cuff blood pressure (from vitalaperiodic) =====
# We already have this in `aperiodic_raw`
# Just rename columns to match the merge later

cuff_bp = aperiodic_raw.rename(columns={
    "noninvasivesystolic": "sbp",
    "noninvasivediastolic": "dbp",
    "noninvasivemean": "map",
})[["patientunitstayid", "observationoffset", "sbp", "dbp", "map"]]

# ===== Continuous (arterial) blood pressure (from vitalperiodic) =====
arterial_bp = vitals_raw.rename(columns={
    "systemicsystolic": "sbp",
    "systemicdiastolic": "dbp",
    "systemicmean": "map",
})[["patientunitstayid", "observationoffset", "sbp", "dbp", "map"]]

# ===== Combine both BP sources =====
# Stack arterial + cuff measurements; aggregation will use both
all_bp = pd.concat([arterial_bp, cuff_bp], ignore_index=True)
print(f"Combined BP measurements: {len(all_bp):,} rows")
print(f"Unique patients with any BP: {all_bp.dropna(subset=['sbp', 'dbp', 'map'], how='all')['patientunitstayid'].nunique()}")

# ===== Temperature from nursecharting (Celsius) =====
temp_query = text("""
    SELECT 
        nc.patientunitstayid,
        nc.nursingchartoffset AS observationoffset,
        nc.nursingchartvalue::float AS temperature
    FROM nursecharting nc
    INNER JOIN cohort c ON nc.patientunitstayid = c.patientunitstayid
    WHERE 
        nc.nursingchartoffset >= 0
        AND nc.nursingchartoffset <= 1440
        AND nc.nursingchartcelltypevalname = 'Temperature (C)'
        AND nc.nursingchartvalue ~ '^-?[0-9]+\\.?[0-9]*$'  -- numeric only
""")

print("\nQuerying nursecharting temperature...")
with engine.connect() as conn:
    nursing_temp = pd.read_sql(temp_query, conn)

# Continuous temperature from vitalperiodic
continuous_temp = vitals_raw[["patientunitstayid", "observationoffset", "temperature"]].dropna(subset=["temperature"])

# Combine
all_temp = pd.concat([continuous_temp, nursing_temp], ignore_index=True)
print(f"Combined temperature measurements: {len(all_temp):,} rows")
print(f"Unique patients with any temperature: {all_temp['patientunitstayid'].nunique()}")

Combined BP measurements: 451,498 rows
Unique patients with any BP: 1367

Querying nursecharting temperature...
Combined temperature measurements: 33,859 rows
Unique patients with any temperature: 1344


## Re-aggregate with the combined data

In [30]:
# Apply plausibility filters
bp_filtered = all_bp[
    (all_bp["sbp"].between(40, 300, inclusive='both') | all_bp["sbp"].isna()) &
    (all_bp["dbp"].between(20, 200, inclusive='both') | all_bp["dbp"].isna()) &
    (all_bp["map"].between(30, 200, inclusive='both') | all_bp["map"].isna())
].copy()

temp_filtered = all_temp[all_temp["temperature"].between(28, 43, inclusive='both')].copy()

# Drop existing BP and temp columns from features
old_cols = [c for c in features.columns 
            if any(c.startswith(p) for p in ["sbp_", "dbp_", "map_", "temp_"])]
features = features.drop(columns=old_cols)

# Aggregate BP (separately for each component because each has its own missingness)
for component in ["sbp", "dbp", "map"]:
    valid = bp_filtered.dropna(subset=[component])[["patientunitstayid", component]]
    agg = (
        valid.groupby("patientunitstayid")[component]
        .agg([
            (f"{component}_min", "min"),
            (f"{component}_max", "max"),
            (f"{component}_mean", "mean"),
            (f"{component}_count", "count"),
        ])
        .reset_index()
    )
    features = features.merge(agg, on="patientunitstayid", how="left")
    features[f"{component}_missing"] = features[f"{component}_mean"].isna()
    n_present = (~features[f"{component}_missing"]).sum()
    print(f"  {component:5s}: {n_present}/1424 patients have data ({100*n_present/1424:.1f}%)")

# Aggregate temperature
temp_agg = (
    temp_filtered.groupby("patientunitstayid")["temperature"]
    .agg([
        ("temp_min", "min"),
        ("temp_max", "max"),
        ("temp_mean", "mean"),
        ("temp_count", "count"),
    ])
    .reset_index()
)
features = features.merge(temp_agg, on="patientunitstayid", how="left")
features["temp_missing"] = features["temp_mean"].isna()
n_present = (~features["temp_missing"]).sum()
print(f"  temp : {n_present}/1424 patients have data ({100*n_present/1424:.1f}%)")

print(f"\nFeatures DataFrame shape: {features.shape}")
print(f"Expected: (1424, 47)")

  sbp  : 1367/1424 patients have data (96.0%)
  dbp  : 1367/1424 patients have data (96.0%)
  map  : 1367/1424 patients have data (96.0%)
  temp : 1341/1424 patients have data (94.2%)

Features DataFrame shape: (1424, 47)
Expected: (1424, 47)


## Sanity check the new values

In [31]:
vital_summary_cols = ["hr_mean", "sbp_mean", "dbp_mean", "map_mean", "rr_mean", "spo2_mean", "temp_mean"]

print("Per-patient mean values across cohort (after BP/temp expansion):")
print(features[vital_summary_cols].describe().T[["count", "mean", "std", "min", "50%", "max"]])

Per-patient mean values across cohort (after BP/temp expansion):
            count        mean        std        min         50%         max
hr_mean    1370.0   85.077344  16.868615  37.000000   83.258935  139.937063
sbp_mean   1367.0  120.544869  18.622470  70.000000  118.416667  191.222222
dbp_mean   1367.0   65.332190  11.593264  34.216495   64.240000  111.521739
map_mean   1367.0   81.626475  12.924584  51.888889   79.791667  137.777778
rr_mean    1259.0   19.758801   4.620154   6.538462   19.014778   45.770642
spo2_mean  1362.0   96.689972   2.147583  77.937238   96.869425  100.000000
temp_mean  1341.0   36.805272   0.557841  32.974931   36.800000   39.800000


In [ ]:
features.to_parquet("../data/features_with_vitals.parquet", index=False)

print(f"Saved features {features.shape} to data/features_with_vitals.parquet")